# News Text Classification Task Using Natural Language Processing

**APOAI2025 Mock Competition Question 3**

The goal of the problem is to read a news article and predict its topic using the given training CSV dataset wherein unlike the basketball problem in question 1, the inputs here are words rather than coordinates. Because of this, the words must be converted into numbers.

# 1. The plan

I will be making the model through word embedding to represent words and a Long Short-Term Memory to process all of them in order. The general flow would be `reading the data, splitting it, creating a vocabulary, train and evaluate the model, then train a final model`.

I also compared different settings in the [seperate extension notebook](../Question3-Extension/experiment.ipynb). This submission uses the selected recipe directly: **192 words, batch size 32, learning rate 0.003, and 25 epochs**.

# 2. Setup

| Setting | Value | What it controls |
| --- | ---: | --- |
| Training epochs | 25 | Passes through the current training articles |
| Batch size | 32 | Maximum articles per weight update |
| Learning rate | 0.003 | Adam's starting step size |
| Seed | 42 | Repeatable initialization and splitting |
| CPU threads | 4 | Threads used for CPU calculations |
| Maximum article length | 192 words | How much text the model reads |
| Vocabulary limit | 12,000 | Maximum number of word-ID entries |
| Embedding / LSTM sizes | 64 / 64 | Number of values used to represent words and sequence outputs |


My initial settings came from an LLM suggestion ([original reference](https://chatgpt.com/share/6ad1a34b-5438-83ec-96a0-d63bff126d1c)). I then compared alternatives using validation macro F1 in the extension notebook and selected the values above. They performed best among the trials but are still not guaranteed to be the best possible settings.

In [12]:
import csv
import json
import random
import re
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import torch
from torch import nn

roots = [Path.cwd(), *Path.cwd().parents]
candidates = ([root / 'ICAIC_Outputs' / 'Question3' for root in roots]
              + [root / 'Question3' for root in roots] + roots)
PART_FOLDER = next(folder for folder in candidates
                   if (folder / 'train_v2' / 'train_news.csv').is_file())
TRAIN_PATH = PART_FOLDER / "train_v2" / "train_news.csv"
TEST_PATH = PART_FOLDER / "train_v2" / "test_news_nolabel.csv"
OUTPUT_DIR = PART_FOLDER

settings = SimpleNamespace(epochs=25, batch_size=32, learning_rate=0.003, seed=42,
                           threads=4, max_length=192, max_vocabulary=12000)
DEDUP_LENGTH = 128
torch.set_num_threads(settings.threads)
torch.use_deterministic_algorithms(True)
print('Training CSV:', TRAIN_PATH)
print('Test CSV:', TEST_PATH)
print('Output folder:', OUTPUT_DIR)


def tokenize(text, max_length):
    return re.findall("[a-z]+(?:'[a-z]+)?", text.lower())[:max_length]


Training CSV: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question3/train_v2/train_news.csv
Test CSV: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question3/train_v2/test_news_nolabel.csv
Output folder: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question3


In [13]:
groups = {}

with TRAIN_PATH.open(newline='', encoding='utf-8-sig') as file:
    for row_number, row in enumerate(csv.DictReader(file), start=1):
        words = tuple(tokenize(row['text'], DEDUP_LENGTH))
        groups.setdefault(words, []).append(
            (row_number, row['category'], row['text'])
        )


duplicates = [group for group in groups.values() if len(group) > 1]
conflicts = [
    group for group in duplicates
    if len({category for _, category, _ in group}) > 1
]

print("Repeated input groups:", len(duplicates))
print("Extra repeated rows:", sum(len(group) - 1 for group in duplicates))
print("Groups with conflicting labels:", len(conflicts))

if conflicts:
    print("\nConflicting rows:")
    for group in conflicts:
        print("\nSame model input, different labels:")
        for row_number, category, text in group:
            print(f"Row {row_number} | Category: {category}")
            print(text[:200])
else:
    print("\nNo conflicting labels found.\n")

with TRAIN_PATH.open(newline='', encoding='utf-8-sig') as file:
    counts = Counter(row['category'] for row in csv.DictReader(file))

for category, count in sorted(counts.items()):
    print(f"{category}: {count} articles")

print(f"Total: {sum(counts.values())} articles")

Repeated input groups: 30
Extra repeated rows: 30
Groups with conflicting labels: 0

No conflicting labels found.

business: 290 articles
entertainment: 205 articles
sport: 284 articles
tech: 221 articles
Total: 1000 articles


# 2. Read the articles

The CSV contains article text and the correct category. I split text into lowercase English words, then assign category IDs: business = 0, entertainment = 1, sport = 2, and tech = 3.

So I can reproduce the experiment, the `seen` dictionary groups articles by their first **128 words** and keeps one copy of each group. This removes 30 repeated inputs which leaves a total of **970 articles** and then for each retained article, I keep up to **192 words** for the model to read.

The duplicate-checking cell also checks whether matching groups have different labels. No conflicting labels were found in the supplied CSV.

In [14]:
def read_training_data(path, max_length):
    with path.open(newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        if not {'text', 'category'}.issubset(reader.fieldnames or []):
            raise ValueError('Training CSV must contain text and category columns')
        rows = list(reader)
    if not rows or any((not row['text'] or not row['category'] for row in rows)):
        raise ValueError('Training rows need nonempty text and category values')
    categories = sorted({row['category'] for row in rows})
    category_ids = {name: index for index, name in enumerate(categories)}
    texts = []
    tokens = []
    labels = []
    seen = {}
    for row in rows:
        words = tokenize(row['text'], max_length)
        if not words:
            raise ValueError('A training article has no usable English words')
        # Keep the same article groups used in the parameter comparison.
        key = tuple(tokenize(row['text'], DEDUP_LENGTH))
        if key in seen:
            if seen[key] != row['category']:
                raise ValueError('Identical model inputs have conflicting categories')
            continue
        seen[key] = row['category']
        texts.append(row['text'])
        tokens.append(words)
        labels.append(category_ids[row['category']])
    return (texts, tokens, torch.tensor(labels), categories, len(rows))

texts, tokens, labels, categories, original_count = read_training_data(TRAIN_PATH, settings.max_length)
print(f'Original CSV rows: {original_count:,}')
print(f'Distinct model inputs: {len(texts):,}')
print(f'Repeated inputs set aside: {original_count - len(texts)}')
print('Category order:', categories)


Original CSV rows: 1,000
Distinct model inputs: 970
Repeated inputs set aside: 30
Category order: ['business', 'entertainment', 'sport', 'tech']


# 3. Split the data

I keep the same stratified split used in the experiments:

```text
970 articles
    ├── 776 development articles
    │     ├── 621 training articles
    │     └── 155 validation articles
    └── 194 holdout articles
```

The extension used training and validation to choose settings. Here I train that fixed recipe, then check it on the same local groups. The holdout has been inspected during earlier work, so it is a reused local check rather than a new untouched test.

After this check, the final saved model is trained on all 970 articles.


In [15]:
def split_data(indices, labels, seed):
    rng = random.Random(seed)
    answers = labels.tolist()
    training = []
    holdout = []
    for label in sorted(set(answers)):
        group = [index for index in indices if answers[index] == label]
        rng.shuffle(group)
        count = round(0.2 * len(group))
        holdout.extend(group[:count])
        training.extend(group[count:])
    rng.shuffle(training)
    rng.shuffle(holdout)
    return (torch.tensor(training), torch.tensor(holdout))

development, holdout = split_data(list(range(len(texts))), labels, settings.seed)
training, validation = split_data(development.tolist(), labels, settings.seed + 1)
print(f'Learn weights: {len(training)}; choose epoch count: {len(validation)}; check: {len(holdout)}')

Learn weights: 621; choose epoch count: 155; check: 194


# 4. Convert words into IDs

Since networks can't receive strings directly, I made it a dictionary that assigns each known word an integer ID. For instance, suppose it has:


```python
{'<PAD>': 0, '<UNK>': 1, 'team': 2, 'won': 3}
```

Then a short article could be encoded as:

```text
['team', 'won'] → [2, 3, 0, 0, ...]
```

ID **0** fills all of the unused positions while ID **1** represents a missing word from the collected vocabulary. `Counter` counts word occurrences wherein words that appear at least twice can receive their own IDS. `Encode` starts with a zero-filled tensor and writes each article's IDs into its row. For 621 training articles then the shape becomes: `(621, 192)`. Note that strictly only the 621 training articles determine this first vocabulary.


In [16]:
def build_vocabulary(token_lists, max_vocabulary):
    counts = Counter((word for words in token_lists for word in words))
    vocabulary = {'<PAD>': 0, '<UNK>': 1}
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    for word, count in ordered:
        if count >= 2 and len(vocabulary) < max_vocabulary:
            vocabulary[word] = len(vocabulary)
    return vocabulary

def encode(token_lists, vocabulary, max_length):
    encoded = torch.zeros(len(token_lists), max_length, dtype=torch.long)
    for index, words in enumerate(token_lists):
        ids = [vocabulary.get(word, 1) for word in words[:max_length]] or [1]
        encoded[index, :len(ids)] = torch.tensor(ids)
    return encoded

training_tokens = [tokens[index] for index in training.tolist()]
training_vocabulary = build_vocabulary(training_tokens, settings.max_vocabulary)
training_features = encode(training_tokens, training_vocabulary, settings.max_length)
validation_features = encode([tokens[index] for index in validation.tolist()], training_vocabulary, settings.max_length)
print('Vocabulary entries for the first training stage:', len(training_vocabulary))
print('Training input shape:', tuple(training_features.shape))


Vocabulary entries for the first training stage: 6757
Training input shape: (621, 192)


# 5. Creating the model

As mentioned, I used a word embedding and an LSTM to predict whether an article is about **business, entertainment, sport, or tech**.

1. The article is converted to lowercase words, keeping the first 192. Each word is replaced with its vocabulary ID. Unknown words use ID `1` and unused positions use padding ID `0` as. explained in the previous step

2. The embedding layer converts each word ID into 64 numbers wherein these numbers are adjusted during training to help the model predict topics.

3. The LSTM then processes the word representations in order, carrying information from earlier words. It produces 64 output values at each position.

4. I average the LSTM outputs for the real words to get one list of 64 numbers for the article. Padding positions are excluded from the average. Separately, dropout temporarily zeros some averaged values during training to reduce reliance on individual values.

5. Last, a linear layer converts the article representation into four scores, one per category and then the highest score determines the predicted topic.


In [17]:
class MyModel(nn.Module):

    def __init__(self, vocabulary_size, number_of_categories):
        super().__init__()
        # One trainable 64-number vector for each vocabulary entry.
        self.embedding = nn.Embedding(vocabulary_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(64, number_of_categories)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        # Keep real-word positions when averaging the LSTM outputs.
        mask = (x != 0).unsqueeze(-1)
        pooled = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        # Dropout operates during training and is disabled by model.eval().
        pooled = self.dropout(pooled)
        return self.fc1(pooled)

example_model = MyModel(len(training_vocabulary), len(categories))
with torch.no_grad():
    example_scores = example_model(training_features[:2])

## 6. Training and evaluation

I shuffle the training articles and process them in batches of 32. The model predicts four scores, cross-entropy compares them with the answers, and Adam adjusts the embedding, LSTM, and linear-layer parameters. Gradient clipping limits unusually large gradients.

The model trains for **25 epochs**, chosen using validation macro F1 in the experiments file. For reference, **Accuracy** is the fraction of articles predicted correctly while **Macro F1** averages the four topic F1 scores equally, considering both false predictions and missed articles of each topic.


In [18]:
@torch.no_grad()
def predict_scores(model, features, batch_size):
    model.eval()
    scores = []
    for start in range(0, len(features), batch_size):
        scores.append(model(features[start:start + batch_size]))
    return torch.cat(scores)


def classification_metrics(labels, predictions, categories):
    f1_scores = []
    for index in range(len(categories)):
        actual = labels == index
        predicted = predictions == index
        tp = (actual & predicted).sum().item()
        fp = (~actual & predicted).sum().item()
        fn = (actual & ~predicted).sum().item()
        denominator = 2 * tp + fp + fn
        f1_scores.append(2 * tp / denominator if denominator else 0.0)

    correct = (labels == predictions).sum().item()
    return {
        'macro_f1': sum(f1_scores) / len(categories),
        'accuracy': (labels == predictions).float().mean().item(),
        'correct': correct,
        'total': len(labels),
    }


def evaluate(model, features, labels, categories, batch_size):
    scores = predict_scores(model, features, batch_size)
    # Select the highest-scoring class separately for each article.
    predictions = scores.argmax(dim=1)
    metrics = classification_metrics(labels, predictions, categories)
    metrics['cross_entropy'] = nn.CrossEntropyLoss()(scores, labels).item()
    return metrics


def fit(features, labels, vocabulary_size, categories, epochs, settings):
    # Start from fresh weights for each training stage.
    torch.manual_seed(settings.seed)
    model = MyModel(vocabulary_size, len(categories))
    optimizer = torch.optim.Adam(
        model.parameters(), lr=settings.learning_rate, weight_decay=0.0001
    )
    loss_function = nn.CrossEntropyLoss()
    generator = torch.Generator().manual_seed(settings.seed + 1000)

    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(features), generator=generator)
        total_loss = 0.0
        for start in range(0, len(features), settings.batch_size):
            batch = order[start:start + settings.batch_size]
            optimizer.zero_grad(set_to_none=True)
            scores = model(features[batch])
            loss = loss_function(scores, labels[batch])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(batch)
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(f'Epoch {epoch}/{epochs}: training loss={total_loss / len(features):.6f}', flush=True)

    model.eval()
    return model


In [19]:
chosen_epoch = settings.epochs
print('Using the epoch count selected in the extension:', chosen_epoch)
validation_model = fit(
    training_features, labels[training], len(training_vocabulary), categories,
    chosen_epoch, settings,
)
validation_metrics = evaluate(
    validation_model, validation_features, labels[validation], categories, settings.batch_size
)
print(f"Validation accuracy: {validation_metrics['accuracy']:.2%}")
print(f"Validation macro F1: {validation_metrics['macro_f1']:.6f}")

Using the epoch count selected in the extension: 25
Epoch 1/25: training loss=1.362462
Epoch 5/25: training loss=0.575149
Epoch 10/25: training loss=0.110350
Epoch 15/25: training loss=0.049596
Epoch 20/25: training loss=0.009642
Epoch 25/25: training loss=0.005476
Validation accuracy: 91.61%
Validation macro F1: 0.911733


# 7. Evaluate the holdout

I rebuild the vocabulary using all **776 development articles** and train a fresh model for the fixed 25 epochs. Note that the **194 holdout articles** use that vocabulary but do not update it or change the weights.

Similar with the previous solutions on the other questions I made a baseline which always predicts the most common development category then compared its accuracy and macro F1 with the model's scores.

In [20]:
development_tokens = [tokens[index] for index in development.tolist()]
evaluation_vocabulary = build_vocabulary(development_tokens, settings.max_vocabulary)
development_features = encode(development_tokens, evaluation_vocabulary, settings.max_length)
holdout_features = encode(
    [tokens[index] for index in holdout.tolist()], evaluation_vocabulary, settings.max_length
)

evaluation_model = fit(
    development_features, labels[development], len(evaluation_vocabulary),
    categories, chosen_epoch, settings
)
holdout_metrics = evaluate(
    evaluation_model, holdout_features, labels[holdout], categories, settings.batch_size
)

# A baseline makes one constant guess based on development labels only.
majority_label = torch.bincount(labels[development], minlength=len(categories)).argmax().item()
majority_predictions = torch.full_like(labels[holdout], majority_label)
baseline_metrics = classification_metrics(labels[holdout], majority_predictions, categories)

print(f"Holdout correct: {holdout_metrics['correct']}/{holdout_metrics['total']}")
print(f"Holdout accuracy: {holdout_metrics['accuracy']:.2%}")
print(f"Holdout macro F1: {holdout_metrics['macro_f1']:.6f}")
print(f"Baseline (always {categories[majority_label]}): "
      f"accuracy={baseline_metrics['accuracy']:.2%}, macro F1={baseline_metrics['macro_f1']:.6f}")


Epoch 1/25: training loss=1.358474
Epoch 5/25: training loss=0.470259
Epoch 10/25: training loss=0.077719
Epoch 15/25: training loss=0.068395
Epoch 20/25: training loss=0.048811
Epoch 25/25: training loss=0.037711
Holdout correct: 159/194
Holdout accuracy: 81.96%
Holdout macro F1: 0.820178
Baseline (always business): accuracy=29.38%, macro F1=0.113546


# 8. Train and save the final model

After the local check, I rebuild the vocabulary and train a fresh model on all of the 970 distinct inputs. Note that the holdout score is not an independent evaluation of these final weights because final training now includes those articles too. Then, the `.pth` file saves the parameters, the `.py` file saves the model structure and the metadata JSON saves the vocabulary and category order.

In [21]:
vocabulary = build_vocabulary(tokens, settings.max_vocabulary)
features = encode(tokens, vocabulary, settings.max_length)
model = fit(features, labels, len(vocabulary), categories, chosen_epoch, settings)
weights_path = OUTPUT_DIR / 'submission_dic.pth'
torch.save(model.state_dict(), weights_path)
metadata = {'vocabulary': vocabulary, 'categories': categories, 'max_length': settings.max_length,
            'selected_epochs': chosen_epoch, 'embedding_size': 64, 'hidden_size': 64}
(OUTPUT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
model_definition = """from torch import nn

class MyModel(nn.Module):

    def __init__(self, vocabulary_size, number_of_categories):
        super().__init__()
        # One trainable 64-number vector for each vocabulary entry.
        self.embedding = nn.Embedding(vocabulary_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(64, number_of_categories)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        # Keep real-word positions when averaging the LSTM outputs.
        mask = (x != 0).unsqueeze(-1)
        pooled = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        # Dropout operates during training and is disabled by model.eval().
        pooled = self.dropout(pooled)
        return self.fc1(pooled)

"""
(OUTPUT_DIR / 'submission_model.py').write_text(model_definition, encoding='utf-8')
print('Saved selected weights, model definition, and vocabulary in:', OUTPUT_DIR)


Epoch 1/25: training loss=1.323265
Epoch 5/25: training loss=0.278986
Epoch 10/25: training loss=0.053380
Epoch 15/25: training loss=0.057328
Epoch 20/25: training loss=0.015470
Epoch 25/25: training loss=0.022048
Saved selected weights, model definition, and vocabulary in: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question3


# 9. Reload and predict the test file

The first two training articles demonstrate loading, with predicted and actual labels shown together.

In [22]:
saved_metadata = json.loads((OUTPUT_DIR / 'model_metadata.json').read_text())
restored_model = MyModel(len(saved_metadata['vocabulary']), len(saved_metadata['categories']))
restored_model.load_state_dict(torch.load(weights_path, map_location='cpu', weights_only=True))
restored_model.eval()
example_ids = predict_scores(restored_model, features[:2], settings.batch_size).argmax(dim=1).tolist()
print('First two predicted topics:', [categories[index] for index in example_ids])
print('Actual topics:', [categories[index] for index in labels[:2].tolist()])

First two predicted topics: ['sport', 'business']
Actual topics: ['sport', 'business']


# 10. Conclusion and limitations

I used the settings selected in the [extension notebook](../Question3-Extension/experiment.ipynb) which gave **192 words, batch size 32, learning rate 0.003, and 25 epochs**. The evaluation model correctly classified **159 of 194 holdout articles**, giving `81.96% accuracy` and **0.820178 macro F1**. The original 8-epoch recipe using an LLM's suggestion had 105 correct which has a `54.12% accuracy` and 0.494077 macro F1.

After checking the recipe, I trained a fresh final model on all **970 articles** and saved its parameters and vocabulary beside this notebook.The validation score was higher than the holdout score so the results are still uncertain on new articles.

## 11. How the experiment informed this submission

The separate [extension notebook](../Question3-Extension/experiments.ipynb) is the experiment record. It compares the original recipe with changes to epoch count, learning rate, batch size, and article length using the same articles and split.

Validation macro F1 selected the recipe used in this submisison notebook wherein this submission trains that one recipe directly instead of rerunning every trial. The original 8-epoch notebook and files are preserved in `../Question3-Extension/original_baseline` for reference.